# ECFFT: Elliptic Curve Fast Fourier Transform

An interactive walkthrough of the ECFFT over the BN-254 base field.

**The problem:** We want fast polynomial evaluation and interpolation (like the FFT), but the BN-254 base field has no roots of unity of large 2-power order. The ECFFT solves this using elliptic curve isogenies.

**The idea:** Replace the squaring map $x \mapsto x^2$ (2-to-1 on roots of unity) with a rational map $\psi(x) = (x-b)^2/x$ (2-to-1 on x-coordinates of curve points).

In [ ]:
from ecfft_algorithms import (
    q, fadd, fsub, fmul, fdiv, fneg, finv, fsqrt, fpow,
    GoodCurve, Point, RationalMap,
    good_isogeny, apply_isogeny, build_isogeny_chain,
    build_fftree, FFTree, poly_eval, lagrange_interpolate,
    build_evaluation_domain, split_domain_with_psi,
)
from ecfft_params_2_20 import params, verify

## 1. The curve and its parameters

We work over the BN-254 base field $\mathbb{F}_q$ with:
$$q = 21888242871839275222246405745257275088696311157297823662689037894645226208583$$

Our "good curve" has the form $E: y^2 = x^3 + a \cdot x^2 + B \cdot x$ with $B = b^2$.

The key property: $E(\mathbb{F}_q)$ contains a cyclic subgroup of order $2^{20}$.

In [ ]:
print(f"Field: F_q where q = {q}")
print(f"q mod 4 = {q % 4}  (so q ≡ 3 mod 4 — Tonelli-Shanks simplifies to x^((q+1)/4))")
print(f"")
print(f"Curve parameters:")
print(f"  a  = {params['a']}")
print(f"  B  = {params['bb']}")
print(f"  Generator order: 2^{params['k']} = {2**params['k']}")

In [ ]:
# Verify the parameters
verify()

## 2. The good isogeny and the ψ map

A "good isogeny" $\phi: E \to E'$ with kernel $\langle T \rangle$ where $T = (0, 0)$ gives us:

$$\phi(x, y) = \left(\psi(x),\ h(x) \cdot y\right)$$

where $\psi(x) = \frac{(x - b)^2}{x}$ is the x-coordinate map.

**Key property:** $\psi$ is 2-to-1 on the evaluation domain. If $P$ and $P + T$ are in the domain, then $\psi(x(P)) = \psi(x(P + T))$.

In [ ]:
curve = GoodCurve(params['a'], params['bb'])
gen = Point(params['gx'], params['gy'], curve)

# Build the first isogeny
psi, h, codomain = good_isogeny(curve)

print(f"Curve E:  y² = x³ + a·x² + B·x")
print(f"  b = √B = {curve.b}")
print(f"  ψ(x) = (x - b)² / x")
print(f"")
print(f"Codomain E':  y² = x³ + {codomain.a}·x² + {codomain.bb}·x")
print(f"  a' = a + 6b = {fadd(curve.a, fmul(6, curve.b))}")

### ψ is 2-to-1: a concrete example

Let's build a small domain (size 8) and verify that ψ maps it 2-to-1.

In [ ]:
# Scale generator to order 8 = 2^3
k = params['k']
gen8 = gen.scalar_mul(2 ** (k - 3))

# Build evaluation domain: x-coordinates of O+coset, G+coset, 2G+coset, ..., 7G+coset
coset = gen.double()
domain = build_evaluation_domain(gen8, coset, 8)
print(f"Domain (8 x-coordinates):")
for i, x in enumerate(domain):
    print(f"  s[{i}] = {x}")

In [ ]:
# Build the isogeny chain for 3 levels (log2(8) = 3)
psis, curves, hs = build_isogeny_chain(gen8, 3)

# Show ψ₀ is 2-to-1: first half pairs with second half
print("ψ₀ pairing (first half ↔ second half):")
for i in range(4):
    img_first = psis[0](domain[i])
    img_second = psis[0](domain[i + 4])
    print(f"  ψ₀(s[{i}]) = ψ₀(s[{i+4}]) = {img_first}   {'✓' if img_first == img_second else '✗'}")

## 3. The isogeny chain: recursive halving

Each ψ map halves the domain. Starting from 8 points, we get:

$$8 \xrightarrow{\psi_0} 4 \xrightarrow{\psi_1} 2 \xrightarrow{\psi_2} 1$$

This is the ECFFT analogue of the FFT butterfly: at each level, pairs of points collapse to one.

In [ ]:
current = list(domain)
for level in range(3):
    half = len(current) // 2
    s0, s1 = current[:half], current[half:]
    images = [psis[level](s0[i]) for i in range(half)]
    print(f"Level {level}: {len(current)} points → {len(images)} points via ψ_{level}")
    for i in range(half):
        assert psis[level](s0[i]) == psis[level](s1[i])
    current = images
print(f"\nFinal: {len(current)} point(s) — the recursion base case")

## 4. ENTER: Fast polynomial evaluation

The classic FFT decomposes $f(x) = f_{\text{even}}(x^2) + x \cdot f_{\text{odd}}(x^2)$.

The ECFFT decomposes $f(x) = u(\psi(x)) + x^{n/2} \cdot v(\psi(x))$ where $u$ has the low coefficients and $v$ the high.

**ENTER** converts coefficient → evaluation representation in $O(n \log^2 n)$.

In [ ]:
# Build an FFTree of size 8
tree, leaves = build_fftree(params, log_n=3)
eval_domain = tree.eval_domain()

# A polynomial: f(x) = 1 + 2x + 3x² + ... + 8x⁷
coeffs = list(range(1, 9))
print(f"f(x) = {' + '.join(f'{c}x^{i}' if i > 0 else str(c) for i, c in enumerate(coeffs))}")
print()

In [ ]:
# ENTER: fast evaluation
evals = tree.enter(coeffs)

# Compare with naive O(n²) evaluation
naive = [poly_eval(coeffs, x) for x in eval_domain]

print("ENTER results (first 4):")
for i in range(4):
    status = '✓' if evals[i] == naive[i] else '✗'
    print(f"  f(s[{i}]) = {evals[i]}  {status}")
print(f"\nAll match naive: {evals == naive}")

## 5. EXIT: Fast polynomial interpolation

**EXIT** is the inverse of ENTER: given evaluations, recover the coefficients.

Algorithm: modular-reduce to get $u = f \bmod x^{n/2}$, recurse, then recover $v = (f - u)/x^{n/2}$.

In [ ]:
# EXIT: fast interpolation
recovered = tree.exit(evals)

print("EXIT round-trip:")
for i in range(8):
    status = '✓' if recovered[i] == coeffs[i] else '✗'
    print(f"  c[{i}] = {recovered[i]}  (expected {coeffs[i]})  {status}")
print(f"\nPerfect round-trip: {recovered == coeffs}")

## 6. DEGREE and EXTEND

Two more key operations:

- **DEGREE**: determine the degree of a polynomial from its evaluation table in $O(n \log n)$
- **EXTEND**: given evaluations on one "moiety" (half of the domain), compute evaluations on the other half

In [ ]:
# DEGREE
deg = tree.degree(evals)
print(f"Degree of f: {deg}  (expected 7 since f has 8 nonzero coefficients)")

# What about a lower-degree polynomial embedded in the same domain?
low_coeffs = [1, 2, 3] + [0] * 5  # degree 2, padded to size 8
low_evals = tree.enter(low_coeffs)
low_deg = tree.degree(low_evals)
print(f"Degree of 1 + 2x + 3x²: {low_deg}  (expected 2)")

In [ ]:
# EXTEND: given evaluations on S₀ (even-indexed domain points),
# compute evaluations on S₁ (odd-indexed points).
# This works for polynomials of degree < n/2.

tree16, _ = build_fftree(params, log_n=4)  # size 16
dom16 = tree16.eval_domain()
s0_points = dom16[0::2]  # 8 even-indexed points
s1_points = dom16[1::2]  # 8 odd-indexed points

# Polynomial of degree 7 (< 16/2 = 8)
poly7 = list(range(1, 9))  # degree 7
s0_vals = [poly_eval(poly7, x) for x in s0_points]

# Extend from S₀ to S₁
s1_extended = tree16.extend(s0_vals, 'S1')
s1_naive = [poly_eval(poly7, x) for x in s1_points]

print(f"EXTEND S₀ → S₁ (degree-7 polynomial on 16-point domain):")
print(f"  Matches naive: {s1_extended == s1_naive}")

## 7. Scaling up

The whole point is that this scales. Let's try a larger domain.

In [ ]:
import time

for log_n in [3, 4, 5, 6]:
    n = 1 << log_n
    tree_n, _ = build_fftree(params, log_n)
    dom_n = tree_n.eval_domain()
    coeffs_n = list(range(1, n + 1))

    t0 = time.time()
    evals_n = tree_n.enter(coeffs_n)
    t_enter = time.time() - t0

    t0 = time.time()
    recovered_n = tree_n.exit(evals_n)
    t_exit = time.time() - t0

    ok = recovered_n == coeffs_n
    naive_n = [poly_eval(coeffs_n, x) for x in dom_n]
    enter_ok = evals_n == naive_n

    print(f"n={n:4d}  ENTER: {t_enter:.3f}s {'✓' if enter_ok else '✗'}  "
          f"EXIT: {t_exit:.3f}s {'✓' if ok else '✗'}")

## 8. Three curves, three domain sizes

We have parameters for curves with cyclic subgroups of order $2^{18}$, $2^{19}$, and $2^{20}$. The domain size for the FFTree can be any power of 2 up to the subgroup order.

In [ ]:
from ecfft_params_2_18 import params as p18
from ecfft_params_2_19 import params as p19
from ecfft_params_2_20 import params as p20

for name, p in [("2^18", p18), ("2^19", p19), ("2^20", p20)]:
    tree_p, _ = build_fftree(p, log_n=4)
    dom_p = tree_p.eval_domain()
    c = list(range(1, 17))
    e = tree_p.enter(c)
    r = tree_p.exit(e)
    naive_p = [poly_eval(c, x) for x in dom_p]
    print(f"Curve {name}:  ENTER {'✓' if e == naive_p else '✗'}  EXIT {'✓' if r == c else '✗'}  "
          f"DEGREE {tree_p.degree(e)}")